# Stage 2 -- Dataset Synthesis

Saves per-dataset cleaned parquets to `data/processed/<lang>/` then merges them into three unified parquets at `data/merged/{c_cpp,java,python}_merged.parquet`.

Steps:
1. **CWE filter** -- keep only `leaf` and `non-leaf` CWEs (drop `category`, `deprecated`, `unknown`). Result saved per dataset under `processed/<lang>/`.
2. **Deduplication** -- SHA-256 of normalised code; higher-quality source wins (`real > synth > ai`); label conflicts are logged. Result saved to `merged/<lang>_merged.parquet`.

**Prerequisites:** run `00_download_datasets.ipynb` first.

In [ ]:
import sys
import hashlib
import re
from pathlib import Path
from collections import Counter

import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW        = ROOT / 'data' / 'raw'
CWE_XML    = ROOT / 'data' / 'cwec_latest.xml'
PROC_DIR   = ROOT / 'data' / 'processed'
MERGED_DIR = ROOT / 'data' / 'merged'
PROC_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

BRANCH_PRIORITY = {'real': 0, 'synth': 1, 'ai': 2}

_LANG_SLUG = {'C/C++': 'c_cpp', 'Java': 'java', 'Python': 'python'}


def _normalize_code(code: str) -> str:
    code = code.replace('\r\n', '\n').replace('\r', '\n')
    lines = [l.rstrip() for l in code.split('\n')]
    while lines and not lines[0].strip():
        lines.pop(0)
    while lines and not lines[-1].strip():
        lines.pop()
    return '\n'.join(lines)


def _code_hash(code: str) -> str:
    return hashlib.sha256(_normalize_code(code).encode('utf-8')).hexdigest()


print(f'Root:      {ROOT}')
print(f'Processed: {PROC_DIR}')
print(f'Merged:    {MERGED_DIR}')

## 1. CWE Navigator

In [ ]:
import io, urllib.request, zipfile
from ingestion.cwe_navigator import CWENavigator

CWE_ZIP_URL = 'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'
if not CWE_XML.exists():
    print('Downloading CWE XML ...')
    with urllib.request.urlopen(CWE_ZIP_URL, timeout=60) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        xml_name = next(n for n in zf.namelist() if n.endswith('.xml'))
        CWE_XML.write_bytes(zf.read(xml_name))

nav = CWENavigator(str(CWE_XML))
print(f'Loaded: {len(nav.weaknesses):,} weaknesses, {len(nav.categories):,} categories')

_parent_ids: set[str] = {p for parents in nav.child_of.values() for p in parents}


def _strip(cwe_str: str) -> str:
    return cwe_str.removeprefix('CWE-').removeprefix('cwe-').strip()


def cwe_label(cwe_str: str) -> str:
    num = _strip(cwe_str)
    if num in nav.categories or num in nav.views:
        return 'category'
    if num not in nav.weaknesses:
        return 'unknown'
    if nav.get_element_name(num).startswith('DEPRECATED:'):
        return 'deprecated'
    return 'leaf' if num not in _parent_ids else 'non-leaf'

## 2. Synthesis builder

In [ ]:
from ingestion.schema import FunctionSample


def _try(name: str, loader):
    try:
        s = loader()
        print(f'  {name:24s}: {len(s):>8,} samples')
        return s
    except FileNotFoundError as e:
        print(f'  {name:24s}: SKIPPED -- {e}')
        return []


def _slugify(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')


def _to_df(name: str, samples: list[FunctionSample], language: str) -> pd.DataFrame:
    """CWE-filter samples and convert to DataFrame."""
    rows = []
    for s in samples:
        if s.label == 1:
            valid_cwes = [c for c in s.cwes if cwe_label(c) in ('leaf', 'non-leaf')]
            if not valid_cwes:
                continue
        else:
            valid_cwes = []
        h = _code_hash(s.code)
        rows.append({
            'code':      s.code,
            'language':  language,
            'label':     s.label,
            'cwes':      valid_cwes,
            'branch':    s.branch,
            'source':    name,
            'code_hash': h,
            'sample_id': s.sample_id or h[:16],
        })
    return pd.DataFrame(rows) if rows else pd.DataFrame(
        columns=['code', 'language', 'label', 'cwes', 'branch', 'source', 'code_hash', 'sample_id'])


def _save_processed(
    collections: dict[str, list[FunctionSample]],
    language: str,
) -> dict[str, pd.DataFrame]:
    """CWE-filter each collection and save per-dataset parquets to processed/<lang>/."""
    lang_dir = PROC_DIR / _LANG_SLUG[language]
    lang_dir.mkdir(parents=True, exist_ok=True)
    dfs: dict[str, pd.DataFrame] = {}
    for name, samples in collections.items():
        df = _to_df(name, samples, language)
        out = lang_dir / f'{_slugify(name)}.parquet'
        df.to_parquet(out, index=False)
        n_v = int((df['label'] == 1).sum())
        n_s = int((df['label'] == 0).sum())
        print(f'  {name:<24}: {len(df):>7,} ({n_v:,}v + {n_s:,}s)  -> {out.relative_to(ROOT)}')
        dfs[name] = df
    return dfs


def _merge(dfs: dict[str, pd.DataFrame], language: str) -> pd.DataFrame:
    """Deduplicate by code hash (real > synth > ai) and save to merged/<lang>_merged.parquet."""
    combined = pd.concat(list(dfs.values()), ignore_index=True)
    combined['_prio'] = combined['branch'].map(lambda b: BRANCH_PRIORITY.get(b, 99))
    combined = combined.sort_values('_prio').reset_index(drop=True)

    conflicts = combined.groupby('code_hash').filter(lambda g: g['label'].nunique() > 1)
    if len(conflicts):
        print(f'  Label conflicts : {conflicts["code_hash"].nunique():>6}  (same code, different label -- first kept)')

    df = (combined.drop_duplicates(subset='code_hash', keep='first')
                  .drop(columns=['_prio'])
                  .reset_index(drop=True))

    n_dup  = len(combined) - len(df)
    n_vuln = int((df['label'] == 1).sum())
    n_safe = int((df['label'] == 0).sum())

    out = MERGED_DIR / f'{_LANG_SLUG[language]}_merged.parquet'
    df.to_parquet(out, index=False)

    print(f'  Input    : {len(combined):>8,}')
    print(f'  Deduped  : {n_dup:>8,}')
    print(f'  Merged   : {len(df):>8,}  ({n_vuln:,} vuln + {n_safe:,} safe)')
    print(f'  Saved    : {out.relative_to(ROOT)}')
    return df

## 3. C/C++

In [ ]:
from ingestion.primevul   import extract_primevul
from ingestion.icvul      import extract_icvul
from ingestion.cvefixes   import extract_cvefixes
from ingestion.megavul    import extract_megavul
from ingestion.secvuleval import extract_secvuleval
from ingestion.crossvul   import extract_crossvul
from ingestion.sven       import extract_sven
from ingestion.juliet     import extract_juliet
from ingestion.castle     import extract_castle
from ingestion.llmseceval import extract_llmseceval


def _load_icvul():
    icvul_dir = RAW / 'icvul'
    if not icvul_dir.exists():
        raise FileNotFoundError(f'icvul/ not found in {RAW}')
    csv = next(icvul_dir.rglob('function_info.csv'), None)
    if csv is None:
        raise FileNotFoundError(f'function_info.csv not found under {icvul_dir}')
    return extract_icvul(csv.parent)


print('Loading C/C++ datasets ...')
collections_c: dict[str, list[FunctionSample]] = {}
collections_c['PrimeVul']      = _try('PrimeVul',      lambda: extract_primevul(RAW/'primevul_train.jsonl') + extract_primevul(RAW/'primevul_test.jsonl'))
collections_c['ICVul']         = _try('ICVul',         _load_icvul)
collections_c['CVEfixes(C)']   = _try('CVEfixes(C)',   lambda: extract_cvefixes(RAW/'cvefixes', language='C'))
collections_c['MegaVul']       = _try('MegaVul',       lambda: extract_megavul(RAW/'megavul'))
collections_c['SecVulEval']    = _try('SecVulEval',    lambda: extract_secvuleval(RAW/'secvuleval.csv'))
collections_c['CrossVul(C)']   = _try('CrossVul(C)',   lambda: extract_crossvul(RAW/'crossvul', language='C/C++'))
collections_c['SVEN(C)']       = _try('SVEN(C)',       lambda: extract_sven(RAW/'sven', language='C/C++'))
collections_c['Juliet(C)']     = _try('Juliet(C)',     lambda: extract_juliet(RAW/'juliet_c.zip', language='C/C++'))
collections_c['CASTLE']        = _try('CASTLE',        lambda: extract_castle(RAW/'castle'/'datasets'))
collections_c['LLMSecEval(C)'] = _try('LLMSecEval(C)', lambda: extract_llmseceval(RAW/'llmseceval', language='C/C++'))
collections_c = {k: v for k, v in collections_c.items() if v}

In [ ]:
print('=== C/C++ -- processed ===')
dfs_c = _save_processed(collections_c, 'C/C++')
print('\n=== C/C++ -- merged ===')
df_c = _merge(dfs_c, 'C/C++')

## 4. Java

In [ ]:
from ingestion.owasp_benchmark import extract_owasp_benchmark
from ingestion.capec_llm       import extract_capec_llm

print('Loading Java datasets ...')
collections_java: dict[str, list[FunctionSample]] = {}
collections_java['CVEfixes(Java)']  = _try('CVEfixes(Java)',  lambda: extract_cvefixes(RAW/'cvefixes', language='Java'))
collections_java['CrossVul(Java)']  = _try('CrossVul(Java)',  lambda: extract_crossvul(RAW/'crossvul', language='Java'))
collections_java['Juliet(Java)']    = _try('Juliet(Java)',    lambda: extract_juliet(RAW/'juliet_java.zip', language='Java'))
collections_java['OWASP(Java)']     = _try('OWASP(Java)',     lambda: extract_owasp_benchmark(RAW/'owasp_benchmark', language='Java'))
collections_java['CAPEC_LLM(Java)'] = _try('CAPEC_LLM(Java)', lambda: extract_capec_llm(RAW/'capec_llm', language='Java'))
collections_java = {k: v for k, v in collections_java.items() if v}

In [ ]:
print('=== Java -- processed ===')
dfs_java = _save_processed(collections_java, 'Java')
print('\n=== Java -- merged ===')
df_java = _merge(dfs_java, 'Java')

## 5. Python

In [ ]:
from ingestion.patcheval        import extract_patcheval
from ingestion.pyvul            import extract_pyvul
from ingestion.sven             import extract_sven
from ingestion.llmseceval       import extract_llmseceval
from ingestion.security_eval    import extract_security_eval

print('Loading Python datasets ...')
collections_py: dict[str, list[FunctionSample]] = {}
collections_py['CVEfixes(Python)']   = _try('CVEfixes(Python)',   lambda: extract_cvefixes(RAW/'cvefixes', language='Python'))
collections_py['PatchEval']          = _try('PatchEval',          lambda: extract_patcheval(RAW/'patcheval'))
collections_py['CrossVul(Python)']   = _try('CrossVul(Python)',   lambda: extract_crossvul(RAW/'crossvul', language='Python'))
collections_py['PyVul']              = _try('PyVul',              lambda: extract_pyvul(RAW/'pyvul'))
collections_py['SVEN(Python)']       = _try('SVEN(Python)',       lambda: extract_sven(RAW/'sven', language='Python'))
collections_py['OWASP(Python)']      = _try('OWASP(Python)',      lambda: extract_owasp_benchmark(RAW/'owasp_benchmark_python', language='Python'))
collections_py['LLMSecEval']         = _try('LLMSecEval',         lambda: extract_llmseceval(RAW/'llmseceval', language='Python'))
collections_py['SecurityEval']       = _try('SecurityEval',       lambda: extract_security_eval(RAW/'security_eval'))
collections_py['CAPEC_LLM(Python)']  = _try('CAPEC_LLM(Python)',  lambda: extract_capec_llm(RAW/'capec_llm', language='Python'))
collections_py = {k: v for k, v in collections_py.items() if v}

In [ ]:
print('=== Python -- processed ===')
dfs_py = _save_processed(collections_py, 'Python')
print('\n=== Python -- merged ===')
df_py = _merge(dfs_py, 'Python')

## 6. Summary

In [ ]:
print(f'{'Language':<10} {'Total':>8} {'Vuln':>8} {'Safe':>8} {'Unique CWEs':>12} {'Sources':>8}')
print('-' * 60)
for lang, df in [('C/C++', df_c), ('Java', df_java), ('Python', df_py)]:
    if df is None or len(df) == 0:
        print(f'{lang:<10} -- no data --')
        continue
    n_vuln = int((df['label'] == 1).sum())
    n_safe = int((df['label'] == 0).sum())
    all_cwes = {c for cwes in df['cwes'] for c in cwes}
    n_sources = df['source'].nunique()
    print(f'{lang:<10} {len(df):>8,} {n_vuln:>8,} {n_safe:>8,} {len(all_cwes):>12,} {n_sources:>8}')

print()
for lang, df in [('C/C++', df_c), ('Java', df_java), ('Python', df_py)]:
    if df is None or len(df) == 0:
        continue
    print(f'--- {lang} by branch ---')
    for branch, grp in df.groupby('branch'):
        n_v = int((grp['label'] == 1).sum())
        n_s = int((grp['label'] == 0).sum())
        print(f'  {branch:<6}: {len(grp):>7,} total  ({n_v:,} vuln + {n_s:,} safe)')
    print()